<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week9_instructor_wfh_controls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9 – Instructor demo: adding and removing controls, live

**For live use in the workshop, not a student handout.** Reproduces the module's main causal case study (Bloom et al.'s Ctrip work-from-home RCT, `wfh_person.csv`, 249 employees) and lets you add or remove controls live to make the confounder-vs-bad-control distinction tangible rather than asserted from a static slide.

**The two headline results**, exactly as on the slide, reproduced here so you can confirm them before changing anything.

## Setup

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

data_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/wfh_person.csv"
wfh = pd.read_csv(data_path)
wfh.head()

In [ ]:
quit_model = smf.ols("quitjob ~ treatment", data=wfh).fit(cov_type="HC1")
print(f"Quit-job effect: {quit_model.params['treatment']*100:.1f}pp, p={quit_model.pvalues['treatment']:.4f}")

ordertakers = wfh[wfh["ordertaker"] == True]
phone_model = smf.ols("phonecalls1 ~ treatment", data=ordertakers).fit(cov_type="HC1")
print(f"Phone-calls effect: {phone_model.params['treatment']:.2f}, p={phone_model.pvalues['treatment']:.4f}")

## Live: does adding pre-treatment covariates move the estimate?

Because `treatment` was **randomly assigned**, pre-treatment characteristics (measured *before* the experiment started) shouldn't be confounders – they should be roughly balanced between treatment and control by design, so adding them as controls should barely move the treatment coefficient.

**Edit the covariate list below and re-run** to test this live with whatever variables the class suggests (available pre-treatment covariates include: `age`, `married`, `children`, `tenure`, `perform10`, `male`, `university`, `costofcommute`).

In [ ]:
COVARIATES = ['age', 'married', 'children', 'tenure', 'perform10']  # <-- edit and re-run

formula = "quitjob ~ treatment + " + " + ".join(COVARIATES)
model_with_controls = smf.ols(formula, data=wfh).fit(cov_type="HC1")

print(f"Treatment only:            {quit_model.params['treatment']:.4f}")
print(f"+ pre-treatment covariates: {model_with_controls.params['treatment']:.4f}")

**Talking point:** the coefficient should barely move (a few percentage points at most, same sign and significance). That's the whole point of randomisation – it's not that these variables don't matter for quitting, it's that they're *unrelated to treatment assignment*, so leaving them out doesn't bias anything. Contrast this with Week 6's OVB demo, where adding the missing variable moved the coefficient substantially – randomisation is precisely what makes that kind of movement disappear.

## Live: what happens if you control for something measured *after* treatment?

`perform11` is a performance score measured **after** the experiment started – a post-treatment outcome, not a pre-treatment characteristic. If working from home itself affects performance, `perform11` sits on the causal pathway (or is otherwise a consequence of treatment) – controlling for it is the "bad control" mistake from this week's DAG section, not a legitimate confounder adjustment.

In [ ]:
wfh_perf = wfh.dropna(subset=['perform11'])

quit_model_samesample = smf.ols("quitjob ~ treatment", data=wfh_perf).fit(cov_type="HC1")
model_bad_control = smf.ols("quitjob ~ treatment + perform11", data=wfh_perf).fit(cov_type="HC1")

print(f"Treatment only:                    {quit_model_samesample.params['treatment']:.4f}")
print(f"+ post-treatment perform11 (bad control): {model_bad_control.params['treatment']:.4f}")

**Talking point:** this one *does* move – noticeably more than the pre-treatment covariates did. Controlling for a post-treatment variable can absorb part of the very effect you're trying to measure, biasing the estimate toward zero. Ask the class: is this the same mechanism as omitted variable bias from Week 6? (No – OVB is bias from *leaving something out*; this is bias *introduced by putting something in*.)